# Anime Recommendation System
**Objective:** To implement a content-based recommendation system using Cosine Similarity.

**Dataset:** Anime Dataset

In [26]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')  


df = pd.read_csv('D:\\DS Assignment\\Recommendation System\\Recommendation System\\anime.csv') 

print("Shape:", df.shape)
display(df.head())
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())

Shape: (12294, 7)


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Military, Shounen",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, Sci-Fi, Shounen",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, Sci-Fi, Shounen",TV,51,9.16,151266



Data Types:
 anime_id      int64
name         object
genre        object
type         object
episodes     object
rating      float64
members       int64
dtype: object

Missing Values:
 anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64


In [22]:
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler


df = df.drop_duplicates(subset='anime_id').reset_index(drop=True)


df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['rating'] = df['rating'].fillna(df['rating'].mean())

df['type'] = df['type'].fillna('Unknown')


def split_genres(x):
    if pd.isna(x): return []
    return [g.strip() for g in x.split(',') if g.strip()!='']

df['genres_list'] = df['genre'].apply(split_genres)


mlb = MultiLabelBinarizer()
genres_mat = mlb.fit_transform(df['genres_list'])

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
type_mat = ohe.fit_transform(df[['type']])


numeric = df[['episodes', 'rating', 'members']].fillna(0)
scaler = StandardScaler()
numeric_scaled = scaler.fit_transform(numeric)


features = np.hstack([genres_mat, type_mat, numeric_scaled])

print("Final Features Shape:", features.shape)

Final Features Shape: (12294, 53)


In [23]:

from sklearn.metrics.pairwise import cosine_similarity


cosine_sim = cosine_similarity(features, features)


indices = pd.Series(df.index, index=df['name']).drop_duplicates()

def get_recommendations(title, sim_threshold=0.5, top_n=10):
    if title not in indices:
        return [], []  
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    
    sim_scores = [(i, score) for i, score in sim_scores if score >= sim_threshold]
    
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    
    sim_scores = [i for i in sim_scores if i[0] != idx][:top_n]
    
    anime_indices = [i[0] for i in sim_scores]
    anime_scores = [i[1] for i in sim_scores]
    
   
    titles_list = df['name'].iloc[anime_indices].tolist()
    
    return titles_list, anime_scores

### Experimenting with Similarity Thresholds
Here we test how changing the similarity threshold affects the number of recommendations.
- **High Threshold (0.8):** Very similar anime, fewer results.
- **Low Threshold (0.5):** More results, but might be less relevant.

In [24]:

test_title = "Naruto"


if test_title not in indices:
    print(f"Title '{test_title}' not found in dataset.")
else:
    print(f"Recommendations for: {test_title}\n")

    for thresh in [0.9, 0.7, 0.5]:
        titles, scores = get_recommendations(test_title, sim_threshold=thresh, top_n=5)
        count = len(titles)
        
        print(f"Threshold {thresh}: Found {count} recommendations.")
        
        if count > 0:
            
            print(f"  Top Pick: {titles[0]} (Sim: {scores[0]:.2f})")
        else:
            print("  No recommendations found above this threshold.")
        print("-" * 30)

Recommendations for: Naruto

Threshold 0.9: Found 5 recommendations.
  Top Pick: Fairy Tail (Sim: 0.98)
------------------------------
Threshold 0.7: Found 5 recommendations.
  Top Pick: Fairy Tail (Sim: 0.98)
------------------------------
Threshold 0.5: Found 5 recommendations.
  Top Pick: Fairy Tail (Sim: 0.98)
------------------------------


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
import numpy as np


train_idx, test_idx = train_test_split(range(len(df)), test_size=0.2, random_state=42)

train_features = features[train_idx]
test_features = features[test_idx]


nn = NearestNeighbors(n_neighbors=11, metric='cosine', algorithm='brute')
nn.fit(train_features)


precisions, recalls, f1s = [], [], []
K = 5

print("Starting Evaluation on Test Set...")

for i in range(min(100, len(test_idx))): 
    test_ind = test_idx[i]
    true_genres = genres_mat[test_ind]
    
    
    distances, indices = nn.kneighbors([test_features[i]], n_neighbors=K+1)
    rec_indices = indices.flatten()[1:] 
    
    
    num_relevant = 0
    for rec_idx in rec_indices:
        
        original_idx = train_idx[rec_idx] 
        if np.dot(true_genres, genres_mat[original_idx]) > 0:
            num_relevant += 1
            
    prec = num_relevant / K
    rec = num_relevant / (K * 2) 
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
    
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)

print(f"Evaluated Samples: {len(precisions)}")
print(f"Avg Precision@{K}: {np.mean(precisions):.4f}")
print(f"Avg Recall@{K}: {np.mean(recalls):.4f}")
print(f"Avg F1-Score@{K}: {np.mean(f1s):.4f}")

Starting Evaluation on Test Set...
Evaluated Samples: 100
Avg Precision@5: 0.9880
Avg Recall@5: 0.4940
Avg F1-Score@5: 0.6587


## Interview Questions & Answers

**1. Can you explain the difference between user-based and item-based collaborative filtering?**
*   **User-Based:** Finds users similar to the target user and recommends items those similar users liked. (e.g., "Users like you also watched...")
*   **Item-Based:** Finds items similar to the ones the target user liked and recommends those similar items. (e.g., "Because you watched Naruto, you might like Bleach.")
*   **Difference:** User-based focuses on user similarity matrices, while item-based focuses on item similarity matrices. Item-based is generally more stable as item preferences change less than user behavior.

**2. What is collaborative filtering, and how does it work?**
*   **Definition:** Collaborative Filtering is a technique used by recommendation systems that predicts a user's interests by collecting preferences from many users.
*   **How it works:** It assumes that if User A and User B agreed on some items in the past, they will likely agree on other items in the future. It does not require information about the item content (like genre), only user-item interaction data (ratings, views).